# Transfer Learning: Model3 (ThesisDNN + XGBoost)

**4 experiments, SEED=42, target fine-tune n=1,000 (fair comparison)**

| Variable | Direction | Target |
|---|---|---|
| `cross_A` | DS-1 → DS-2 cross-transfer | Dataset 2 |
| `cross_B` | DS-2 → DS-1 cross-transfer | Dataset 1 |
| `scratch_D1` | No-transfer DS-1 scratch | Dataset 1 |
| `scratch_D2` | No-transfer DS-2 scratch | Dataset 2 |

**Figures:**
- `transfer_tgt_ds1.png` — Target = Dataset 1 (Blue: cross_B, Red: scratch_D1)
- `transfer_tgt_ds2.png` — Target = Dataset 2 (Blue: cross_A, Red: scratch_D2)

In [24]:
# ─────────────────────────────────────────────────────────────
# File Paths  (both CSVs must be in the same directory as this notebook)
# ─────────────────────────────────────────────────────────────
SOURCE_ORIG  = "original_preprocess.csv"        # Dataset 1
SOURCE_CLEAN = "clean_ul_with_conditions2.csv"  # Dataset 2

OUTPUT_FIG_DS1 = "transfer_tgt_ds1.png"
OUTPUT_FIG_DS2 = "transfer_tgt_ds2.png"

In [25]:
import os, random, copy, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

print("Imports OK")

Imports OK


In [27]:
# ─────────────────────────────────────────────────────────────
# Global Config
# ─────────────────────────────────────────────────────────────
TARGET_COL     = "pm_power"
SEED           = 42
MIN_SLICE_SIZE = 20

# DNN hyper-params (source training)
DNN_EPOCHS  = 400
DNN_PATIENCE= 40
DNN_LR      = 1e-3
DNN_BATCH   = 32
DNN_WD      = 0.01

# Fine-tune hyper-params (freeze fc1–fc4, adapt bottleneck+head only)
FT_EPOCHS   = 50
FT_PATIENCE = 15
FT_LR       = 5e-4
FT_BATCH    = 32

# XGBoost hyper-params
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
XGB_PARAMS = dict(
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    eta              = 0.22,
    max_depth        = 5,
    subsample        = 0.85,
    colsample_bytree = 0.9,
    min_child_weight = 3,
    reg_lambda       = 1.0,
    reg_alpha        = 0.1,
    seed             = SEED,
    tree_method      = "hist",
    device           = DEVICE,
)
XGB_ROUNDS = 256
XGB_EARLY  = 30

print(f"[Config] device={DEVICE} | seed={SEED} ")

[Config] device=cuda | seed=42 


In [28]:
# ─────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()

# ─────────────────────────────────────────────────────────────
# Column Mapping  (original_preprocess _ul suffix → clean naming)
# ─────────────────────────────────────────────────────────────
ORIG_TO_CLEAN = {
    "txgain_ul":       "txgain",
    "selected_mcs_ul": "selected_mcs",
    "airtime_ul":      "airtime",
    "nRBs_ul":         "nRBs",
    "mean_snr_ul":     "mean_snr",
    "bler_ul":         "bler",
    "thr_ul":          "thr",
    "bsr_ul":          "bsr",
}

# ─────────────────────────────────────────────────────────────
# Feature Engineering
# ─────────────────────────────────────────────────────────────
def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    def has(*cols): return all(c in d.columns for c in cols)
    if has("txgain", "airtime"):
        d["txgain_x_airtime"] = d["txgain"] * d["airtime"]
    if has("selected_mcs", "airtime"):
        d["mcs_x_airtime"]    = d["selected_mcs"] * d["airtime"]
    if has("mean_snr", "bler"):
        d["snr_per_bler"]     = d["mean_snr"].astype(float) / (d["bler"].astype(float) + 1e-6)
    if has("thr", "airtime"):
        d["thr_per_airtime"]  = d["thr"].astype(float) / (d["airtime"].astype(float).clip(lower=0.01) + 1e-6)
    return d

In [30]:
# ─────────────────────────────────────────────────────────────
# Feature & Experiment Definitions
# ─────────────────────────────────────────────────────────────
COMMON_BASE = ["txgain", "selected_mcs", "airtime", "nRBs",
               "mean_snr", "bler", "thr", "bsr",
               "turbodec_it", "dec_time", "num_ues"]
COMMON_ENG  = ["txgain_x_airtime", "mcs_x_airtime", "snr_per_bler", "thr_per_airtime"]

EXPERIMENTS = {
    "Transmission Gain": {"slice_col": "txgain",       "base_feats": ["selected_mcs", "airtime", "nRBs"]},
    "MCS":               {"slice_col": "selected_mcs", "base_feats": ["txgain",        "airtime", "nRBs"]},
    "Airtime":           {"slice_col": "airtime",      "base_feats": ["txgain", "selected_mcs", "nRBs"]},
}

def get_feature_cols(exp_cfg, df):
    base  = [c for c in exp_cfg["base_feats"] + COMMON_BASE if c in df.columns]
    eng   = [c for c in COMMON_ENG if c in df.columns]
    feats = list(dict.fromkeys(base + eng))
    return feats

# ─────────────────────────────────────────────────────────────
# Dataset Loaders
# ─────────────────────────────────────────────────────────────
def load_orig(path=SOURCE_ORIG):
    df = pd.read_csv(path)
    df = df.rename(columns=ORIG_TO_CLEAN)
    df = add_feature_engineering(df)
    df = df[df[TARGET_COL] > 0].dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print(f"  [Dataset1/orig]  loaded {len(df):,} rows")
    return df

def load_clean(path=SOURCE_CLEAN):
    df = pd.read_csv(path)
    df = add_feature_engineering(df)
    df = df[df[TARGET_COL] > 0].dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print(f"  [Dataset2/clean] loaded {len(df):,} rows")
    return df

In [31]:
# ─────────────────────────────────────────────────────────────
# Preprocessing Helpers
# ─────────────────────────────────────────────────────────────
def clean_numeric(df, cols):
    d = df.dropna(subset=cols).copy()
    for c in cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d.dropna(subset=cols).copy()

def winsorize_fit(X, lo=1, hi=99):
    X_w, bounds = X.copy().astype(float), []
    for j in range(X.shape[1]):
        l, h = np.percentile(X_w[:, j], lo), np.percentile(X_w[:, j], hi)
        if h > l: X_w[:, j] = np.clip(X_w[:, j], l, h)
        bounds.append((l, h))
    return X_w, bounds

def winsorize_apply(X, bounds):
    X_w = X.copy().astype(float)
    for j, (l, h) in enumerate(bounds):
        if h > l: X_w[:, j] = np.clip(X_w[:, j], l, h)
    return X_w

def split_train_val_test(df, seed=SEED, train_r=0.8, val_r=0.1):
    tr, te = train_test_split(df, test_size=1-train_r, random_state=seed)
    tr, va = train_test_split(tr, test_size=val_r,     random_state=seed)
    return tr, va, te

In [32]:
# ─────────────────────────────────────────────────────────────
# ThesisDNN  (587→261→186→99→bottleneck(16)→head(1))
# ─────────────────────────────────────────────────────────────
class TabularDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.asarray(X), dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.float32).view(-1, 1)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class ThesisDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1        = nn.Linear(input_dim, 587)
        self.fc2        = nn.Linear(587, 261)
        self.fc3        = nn.Linear(261, 186)
        self.fc4        = nn.Linear(186, 99)
        self.bottleneck = nn.Linear(99,  16)
        self.head       = nn.Linear(16,   1)

    def forward(self, x):
        h   = F.relu(self.fc1(x))
        h   = F.relu(self.fc2(h))
        h   = F.relu(self.fc3(h))
        h   = F.relu(self.fc4(h))
        emb = self.bottleneck(h)
        out = self.head(F.relu(emb))
        return out, emb

In [33]:
# ─────────────────────────────────────────────────────────────
# DNN Training
# ─────────────────────────────────────────────────────────────
def train_dnn_generic(X_tr, y_tr, X_va, y_va,
                      input_dim, model=None,
                      epochs=DNN_EPOCHS, batch=DNN_BATCH,
                      lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE,
                      verbose_every=100, seed=SEED):
    set_seed(seed)
    if model is None:
        model = ThesisDNN(input_dim).to(DEVICE)
    else:
        model = model.to(DEVICE)

    loader  = DataLoader(TabularDS(X_tr, y_tr), batch_size=batch, shuffle=True)
    loss_fn = nn.MSELoss()
    # Only optimize parameters with requires_grad=True (handles frozen layers)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr, weight_decay=wd)

    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(DEVICE)
    y_va_t = torch.tensor(y_va, dtype=torch.float32).view(-1, 1).to(DEVICE)

    best_val, best_state, no_impr = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred, _ = model(xb)
            loss_fn(pred, yb).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            vp, _ = model(X_va_t)
            vloss = loss_fn(vp, y_va_t).item()

        if vloss < best_val - 1e-8:
            best_val, best_state, no_impr = vloss, {k: v.cpu().clone()
                for k, v in model.state_dict().items()}, 0
        else:
            no_impr += 1

        if verbose_every and ep % verbose_every == 0:
            print(f"    ep {ep:04d}  val_MSE={vloss:.5f}  no_impr={no_impr}")

        if no_impr >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    return model


@torch.no_grad()
def extract_embeddings(model, X_s, batch_size=1024):
    model.eval()
    X_t  = torch.tensor(X_s, dtype=torch.float32)
    embs = []
    for i in range(0, len(X_t), batch_size):
        _, emb = model(X_t[i:i+batch_size].to(DEVICE))
        embs.append(emb.cpu().numpy())
    return np.vstack(embs)

In [34]:
# ─────────────────────────────────────────────────────────────
# XGBoost Training
# ─────────────────────────────────────────────────────────────
def train_xgb(emb_tr, y_tr, emb_va, y_va,
              xgb_model=None, rounds=XGB_ROUNDS, seed=SEED):
    params  = {**XGB_PARAMS, "seed": seed}
    dtrain  = xgb.DMatrix(emb_tr, label=y_tr)
    dval    = xgb.DMatrix(emb_va, label=y_va)
    booster = xgb.train(
        params, dtrain,
        num_boost_round       = rounds,
        evals                 = [(dval, "val")],
        early_stopping_rounds = XGB_EARLY,
        verbose_eval          = False,
        xgb_model             = xgb_model,
    )
    return booster

In [35]:
# ─────────────────────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────────────────────
def relative_errors(y_true, y_pred, eps=1e-3):
    """Point-wise relative error (%)."""
    yt = np.asarray(y_true).ravel()
    yp = np.asarray(y_pred).ravel()
    return np.abs(yt - yp) / (np.abs(yt) + eps) * 100.0


def slice_re_stats(df_eval, y_pred, slice_col):
    """
    Per-slice (mean_re, max_re, min_re).
    Returns DataFrame: slice_val | mean_re | max_re | min_re
    """
    y_true = df_eval[TARGET_COL].values
    re_arr = relative_errors(y_true, y_pred)
    df_tmp = df_eval[[slice_col]].copy().reset_index(drop=True)
    df_tmp["re"] = re_arr
    agg = (df_tmp.groupby(slice_col)["re"]
                 .agg(mean_re="mean", max_re="max", min_re="min")
                 .reset_index()
                 .rename(columns={slice_col: "slice_val"}))
    return agg

In [36]:
# ─────────────────────────────────────────────────────────────
# run_transfer()  —  SOURCE → TARGET cross-transfer
#
# Steps:
#   1. Train DNN on full source data (all layers)
#   2. Fine-tune DNN on full target train split
#      (fc1–fc4 frozen; only bottleneck+head adapt)
#   3. Extract 16-dim embeddings from fine-tuned DNN
#   4. Train fresh XGBoost on target embeddings
#   5. Evaluate per-slice RE on target test set
#
# Returns: DataFrame [slice_val, mean_re, max_re, min_re]
# ─────────────────────────────────────────────────────────────
def run_transfer(df_src, df_tgt, exp_name, exp_cfg, seed=SEED):
    print(f"\n  ── {exp_name} [cross-transfer] ──")
    slice_col = exp_cfg["slice_col"]

    feat_src = get_feature_cols(exp_cfg, df_src)
    feat_tgt = get_feature_cols(exp_cfg, df_tgt)
    feats    = [f for f in feat_src if f in feat_tgt]
    print(f"    feats ({len(feats)}): {feats}")

    d_src = clean_numeric(df_src, [c for c in feats + [slice_col, TARGET_COL] if c in df_src.columns])
    d_tgt = clean_numeric(df_tgt, [c for c in feats + [slice_col, TARGET_COL] if c in df_tgt.columns])

    # ── Source: full 80/10 split ──────────────────────────────
    s_tr, s_va, _ = split_train_val_test(d_src, seed=seed)
    X_s_tr = s_tr[feats].values.astype(float)
    y_s_tr = s_tr[TARGET_COL].values.astype(float)
    X_s_va = s_va[feats].values.astype(float)
    y_s_va = s_va[TARGET_COL].values.astype(float)

    X_s_tr_w, bounds_s = winsorize_fit(X_s_tr)
    X_s_va_w           = winsorize_apply(X_s_va, bounds_s)
    sc_src             = StandardScaler()
    X_s_tr_s           = sc_src.fit_transform(X_s_tr_w)
    X_s_va_s           = sc_src.transform(X_s_va_w)

    print(f"    [Source DNN] training on {len(s_tr)+len(s_va)} samples …")
    dnn_src = train_dnn_generic(
        X_s_tr_s, y_s_tr, X_s_va_s, y_s_va,
        input_dim=len(feats), epochs=DNN_EPOCHS, batch=DNN_BATCH,
        lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE, seed=seed)

    # ── Target: full 80/10/10 split (no subsampling) ─────────
    t_tr, t_va, t_te = split_train_val_test(d_tgt, seed=seed)
    print(f"    [Target] fine-tune train={len(t_tr)}, val={len(t_va)}, test={len(t_te)}")

    X_t_tr = t_tr[feats].values.astype(float)
    y_t_tr = t_tr[TARGET_COL].values.astype(float)
    X_t_va = t_va[feats].values.astype(float)
    y_t_va = t_va[TARGET_COL].values.astype(float)
    X_t_te = t_te[feats].values.astype(float)

    X_t_tr_w, bounds_t = winsorize_fit(X_t_tr)
    X_t_va_w           = winsorize_apply(X_t_va, bounds_t)
    X_t_te_w           = winsorize_apply(X_t_te, bounds_t)
    sc_tgt             = StandardScaler()
    X_t_tr_s           = sc_tgt.fit_transform(X_t_tr_w)
    X_t_va_s           = sc_tgt.transform(X_t_va_w)
    X_t_te_s           = sc_tgt.transform(X_t_te_w)

    # Fine-tune: freeze fc1–fc4, adapt only bottleneck+head
    dnn_ft = copy.deepcopy(dnn_src)
    for name, param in dnn_ft.named_parameters():
        if name.startswith(("fc1.", "fc2.", "fc3.", "fc4.")):
            param.requires_grad = False
    n_trainable = sum(p.numel() for p in dnn_ft.parameters() if p.requires_grad)
    print(f"    [Fine-tune DNN] {len(t_tr)} samples, trainable params: {n_trainable} …")
    dnn_ft = train_dnn_generic(
        X_t_tr_s, y_t_tr, X_t_va_s, y_t_va,
        input_dim=len(feats), model=dnn_ft,
        epochs=FT_EPOCHS, batch=FT_BATCH,
        lr=FT_LR, wd=DNN_WD, patience=FT_PATIENCE, seed=seed)
    for param in dnn_ft.parameters():
        param.requires_grad = True

    emb_t_tr = extract_embeddings(dnn_ft, X_t_tr_s)
    emb_t_va = extract_embeddings(dnn_ft, X_t_va_s)
    emb_t_te = extract_embeddings(dnn_ft, X_t_te_s)

    print(f"    [Target XGB] training fresh booster …")
    xgb_ft = train_xgb(emb_t_tr, y_t_tr, emb_t_va, y_t_va,
                       xgb_model=None, rounds=XGB_ROUNDS)

    y_pred = xgb_ft.predict(xgb.DMatrix(emb_t_te))
    y_pred = np.clip(y_pred, y_t_tr.min() * 0.9, y_t_tr.max() * 1.1)

    df_eval = t_te.copy().reset_index(drop=True)
    stats   = slice_re_stats(df_eval, y_pred, slice_col)

    re_all = relative_errors(df_eval[TARGET_COL].values, y_pred)
    print(f"    Cross-transfer MRE = {re_all.mean():.2f}%  "
          f"(max={re_all.max():.1f}%  min={re_all.min():.1f}%)")
    return stats

In [37]:
# ─────────────────────────────────────────────────────────────
# run_scratch()  —  No-Transfer Scratch Baseline
#
# Same logic as run_transfer() but WITHOUT transfer:
#   - No source pre-training
#   - DNN trained from random init on full target train split
#   - XGBoost trained from scratch on target embeddings
#   - Same seed + same split as run_transfer() → fair comparison
#
# Returns: DataFrame [slice_val, mean_re, max_re, min_re]
# ─────────────────────────────────────────────────────────────
def run_scratch(df_tgt, exp_name, exp_cfg, seed=SEED):
    print(f"\n  ── {exp_name} [no-transfer scratch] ──")
    slice_col = exp_cfg["slice_col"]

    feats = get_feature_cols(exp_cfg, df_tgt)
    print(f"    feats ({len(feats)}): {feats}")

    d_tgt = clean_numeric(df_tgt, [c for c in feats + [slice_col, TARGET_COL] if c in df_tgt.columns])

    # Same split as run_transfer (same seed → same rows)
    t_tr, t_va, t_te = split_train_val_test(d_tgt, seed=seed)
    print(f"    [Target] scratch train={len(t_tr)}, val={len(t_va)}, test={len(t_te)}")

    X_t_tr = t_tr[feats].values.astype(float)
    y_t_tr = t_tr[TARGET_COL].values.astype(float)
    X_t_va = t_va[feats].values.astype(float)
    y_t_va = t_va[TARGET_COL].values.astype(float)
    X_t_te = t_te[feats].values.astype(float)

    X_t_tr_w, bounds_t = winsorize_fit(X_t_tr)
    X_t_va_w           = winsorize_apply(X_t_va, bounds_t)
    X_t_te_w           = winsorize_apply(X_t_te, bounds_t)
    sc_tgt             = StandardScaler()
    X_t_tr_s           = sc_tgt.fit_transform(X_t_tr_w)
    X_t_va_s           = sc_tgt.transform(X_t_va_w)
    X_t_te_s           = sc_tgt.transform(X_t_te_w)

    # Train DNN from scratch (random init, all layers free)
    print(f"    [Scratch DNN] training from random init on {len(t_tr)} samples …")
    dnn_scratch = train_dnn_generic(
        X_t_tr_s, y_t_tr, X_t_va_s, y_t_va,
        input_dim=len(feats),
        epochs=DNN_EPOCHS, batch=DNN_BATCH,
        lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE,
        seed=seed)

    emb_tr = extract_embeddings(dnn_scratch, X_t_tr_s)
    emb_va = extract_embeddings(dnn_scratch, X_t_va_s)
    emb_te = extract_embeddings(dnn_scratch, X_t_te_s)

    print(f"    [Scratch XGB] training …")
    xgb_scratch = train_xgb(emb_tr, y_t_tr, emb_va, y_t_va,
                            xgb_model=None, rounds=XGB_ROUNDS)

    y_pred = xgb_scratch.predict(xgb.DMatrix(emb_te))
    y_pred = np.clip(y_pred, y_t_tr.min() * 0.9, y_t_tr.max() * 1.1)

    df_eval = t_te.copy().reset_index(drop=True)
    stats   = slice_re_stats(df_eval, y_pred, slice_col)

    re_all = relative_errors(df_eval[TARGET_COL].values, y_pred)
    print(f"    Scratch MRE = {re_all.mean():.2f}%  "
          f"(max={re_all.max():.1f}%  min={re_all.min():.1f}%)")
    return stats

In [38]:
# ─────────────────────────────────────────────────────────────
# Load Datasets
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading datasets …")
print("=" * 60)
df_orig  = load_orig()
df_clean = load_clean()

Loading datasets …
  [Dataset1/orig]  loaded 17,501 rows
  [Dataset2/clean] loaded 5,769 rows


In [39]:
# ─────────────────────────────────────────────────────────────
# Run 4 Experiments  (全量, SEED=42)
#
#   cross_A    — D1→D2 cross-transfer  (source=DS1, target=DS2)
#   cross_B    — D2→D1 cross-transfer  (source=DS2, target=DS1)
#   scratch_D1 — DS-1 no-transfer scratch
#   scratch_D2 — DS-2 no-transfer scratch
#
# Each is a dict indexed by exp_name.
# ─────────────────────────────────────────────────────────────
cross_A    = {}   # D1→D2
cross_B    = {}   # D2→D1
scratch_D1 = {}   # DS-1 scratch
scratch_D2 = {}   # DS-2 scratch

for exp_name, exp_cfg in EXPERIMENTS.items():
    print(f"\n{'='*60}\nExperiment: {exp_name}\n{'='*60}")

    print("\n[1] cross_A — Source=DS-1 → Target=DS-2")
    cross_A[exp_name] = run_transfer(df_orig, df_clean, exp_name, exp_cfg)

    print("\n[2] cross_B — Source=DS-2 → Target=DS-1")
    cross_B[exp_name] = run_transfer(df_clean, df_orig, exp_name, exp_cfg)

    print("\n[3] scratch_D1 — DS-1 no-transfer scratch")
    scratch_D1[exp_name] = run_scratch(df_orig, exp_name, exp_cfg)

    print("\n[4] scratch_D2 — DS-2 no-transfer scratch")
    scratch_D2[exp_name] = run_scratch(df_clean, exp_name, exp_cfg)

print("\n[Done] All 4 experiments finished.")


Experiment: Transmission Gain

[1] cross_A — Source=DS-1 → Target=DS-2

  ── Transmission Gain [cross-transfer] ──
    feats (15): ['selected_mcs', 'airtime', 'nRBs', 'txgain', 'mean_snr', 'bler', 'thr', 'bsr', 'turbodec_it', 'dec_time', 'num_ues', 'txgain_x_airtime', 'mcs_x_airtime', 'snr_per_bler', 'thr_per_airtime']
    [Source DNN] training on 14000 samples …
    [Target] fine-tune train=4153, val=462, test=1154
    [Fine-tune DNN] 4153 samples, trainable params: 1617 …
    [Target XGB] training fresh booster …
    Cross-transfer MRE = 2.43%  (max=21.3%  min=0.0%)

[2] cross_B — Source=DS-2 → Target=DS-1

  ── Transmission Gain [cross-transfer] ──
    feats (15): ['selected_mcs', 'airtime', 'nRBs', 'txgain', 'mean_snr', 'bler', 'thr', 'bsr', 'turbodec_it', 'dec_time', 'num_ues', 'txgain_x_airtime', 'mcs_x_airtime', 'snr_per_bler', 'thr_per_airtime']
    [Source DNN] training on 4615 samples …
    ep 0100  val_MSE=0.08285  no_impr=8
    [Target] fine-tune train=12600, val=1400, tes

In [40]:
# ─────────────────────────────────────────────────────────────
# Figure 1 — Target = Dataset 1 (original_preprocess)
# ─────────────────────────────────────────────────────────────
EXP_NAMES  = list(EXPERIMENTS.keys())
X_LABELS   = {
    "Transmission Gain": "txgain (dBm)",
    "MCS":               "MCS Index",
    "Airtime":           "Airtime Ratio",
}
ROW_LABELS  = ["Relative Error", "Mean Relative Error", "Max Relative Error", "Min Relative Error"]
METRIC_KEYS = ["mean_re",        "mean_re",             "max_re",             "min_re"]

COLOR_BLUE = "#1f77b4"
COLOR_RED  = "#d62728"

fig1, axes1 = plt.subplots(4, 3, figsize=(15, 16), constrained_layout=True)
fig1.suptitle(
    "Transfer Learning — Target: Dataset 1 (original_preprocess) | Model3 (ThesisDNN + XGBoost)",
    fontsize=13, fontweight="bold"
)

for row_idx in range(4):
    mkey   = METRIC_KEYS[row_idx]
    ylabel = f"{ROW_LABELS[row_idx]} (%)"

    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax = axes1[row_idx, col_idx]

        st_blue = cross_B[exp_name].sort_values("slice_val")
        st_red  = scratch_D1[exp_name].sort_values("slice_val")

        x_blue = st_blue["slice_val"].values
        y_blue = st_blue[mkey].values
        x_red  = st_red["slice_val"].values
        y_red  = st_red[mkey].values

        lbl_blue = "Cross-transfer: DS-2\u2192DS-1 (full data)"
        lbl_red  = "No-transfer: DS-1 scratch (full data)"

        ax.plot(x_blue, y_blue, color=COLOR_BLUE, lw=2.2,
                marker="o", ms=4, label=lbl_blue)
        ax.plot(x_red,  y_red,  color=COLOR_RED,  lw=2.2,
                marker="s", ms=4, label=lbl_red)

        ax.set_xlabel(X_LABELS[exp_name], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=8)

        if row_idx == 0:
            ax.set_title(exp_name, fontsize=10, fontweight="bold", pad=6)

        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8.5, loc="upper right",
                      framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_DS1, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_DS1}")
plt.show()

Saved → transfer_tgt_ds1.png


In [41]:
# ─────────────────────────────────────────────────────────────
# Figure 2 — Target = Dataset 2 (clean_ul_with_conditions2)
# ─────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(4, 3, figsize=(15, 16), constrained_layout=True)
fig2.suptitle(
    "Transfer Learning — Target: Dataset 2 (clean_ul_with_conditions2) | Model3 (ThesisDNN + XGBoost)",
    fontsize=13, fontweight="bold"
)

for row_idx in range(4):
    mkey   = METRIC_KEYS[row_idx]
    ylabel = f"{ROW_LABELS[row_idx]} (%)"

    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax = axes2[row_idx, col_idx]

        st_blue = cross_A[exp_name].sort_values("slice_val")
        st_red  = scratch_D2[exp_name].sort_values("slice_val")

        x_blue = st_blue["slice_val"].values
        y_blue = st_blue[mkey].values
        x_red  = st_red["slice_val"].values
        y_red  = st_red[mkey].values

        lbl_blue = "Cross-transfer: DS-1\u2192DS-2 (full data)"
        lbl_red  = "No-transfer: DS-2 scratch (full data)"

        ax.plot(x_blue, y_blue, color=COLOR_BLUE, lw=2.2,
                marker="o", ms=4, label=lbl_blue)
        ax.plot(x_red,  y_red,  color=COLOR_RED,  lw=2.2,
                marker="s", ms=4, label=lbl_red)

        ax.set_xlabel(X_LABELS[exp_name], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=8)

        if row_idx == 0:
            ax.set_title(exp_name, fontsize=10, fontweight="bold", pad=6)

        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8.5, loc="upper right",
                      framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_DS2, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_DS2}")
plt.show()

Saved → transfer_tgt_ds2.png


In [44]:
# ─────────────────────────────────────────────────────────────
# Figure 3 — Metric Heatmap Summary  (2×2 per subplot)
#
# Layout: 3 rows × 3 columns
#   Rows    : Mean Relative Error / Max Relative Error / Min Relative Error
#   Columns : Transmission Gain (Blue) / MCS (Yellow) / Airtime (Green)
# ─────────────────────────────────────────────────────────────
OUTPUT_HEATMAP = "transfer_metric_heatmap.png"

HEATMAP_METRICS = [
    ("mean_re", "mean", "Mean Relative Error (%)"),
    ("max_re",  "max",  "Max Relative Error (%)"),
    ("min_re",  "min",  "Min Relative Error (%)"),
]

# One colormap per column (Transmission Gain / MCS / Airtime)
COL_CMAPS = ["Blues", "YlOrBr", "Greens"]

def scalar_metric(stats_df, col, agg):
    if agg == "mean":  return float(stats_df[col].mean())
    elif agg == "max": return float(stats_df[col].max())
    elif agg == "min": return float(stats_df[col].min())

fig3, axes3 = plt.subplots(3, 3, figsize=(13, 11), constrained_layout=True)
fig3.suptitle(
    "Transfer Learning Metric Summary — Model3 (ThesisDNN + XGBoost)",
    fontsize=13, fontweight="bold"
)

for row_idx, (col_key, agg_fn, metric_label) in enumerate(HEATMAP_METRICS):
    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax   = axes3[row_idx, col_idx]
        cmap = COL_CMAPS[col_idx]

        mat = np.array([
            [scalar_metric(scratch_D1[exp_name], col_key, agg_fn),
             scalar_metric(cross_B[exp_name],    col_key, agg_fn)],
            [scalar_metric(cross_A[exp_name],    col_key, agg_fn),
             scalar_metric(scratch_D2[exp_name], col_key, agg_fn)],
        ])

        im = ax.imshow(mat, cmap=cmap, aspect="auto",
                       vmin=mat.min() * 0.95, vmax=mat.max() * 1.05)

        for ti in range(2):
            for si in range(2):
                val = mat[ti, si]
                text_color = "white" if val > (mat.min() + 0.6 * (mat.max() - mat.min())) else "black"
                ax.text(si, ti, f"{val:.3f}", ha="center", va="center",
                        fontsize=11, fontweight="bold", color=text_color)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(["D1", "D2"], fontsize=9)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["D1", "D2"], fontsize=9)
        ax.set_xlabel("Source", fontsize=9)
        ax.set_ylabel("Target", fontsize=9)
        ax.set_title(f"{metric_label.split(' (')[0]} — {exp_name}", fontsize=9, pad=5)

        cbar = fig3.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=7)

plt.savefig(OUTPUT_HEATMAP, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_HEATMAP}")
plt.show()

Saved → transfer_metric_heatmap.png


In [43]:
# ─────────────────────────────────────────────────────────────
# Summary Table
# ─────────────────────────────────────────────────────────────
print("\n" + "="*95)
print(f"SUMMARY: Cross-transfer vs No-transfer Scratch  (full target dataset, SEED={SEED})")
print("="*95)
print(f"{'Experiment':<22} {'Direction':<40} {'Mean_RE':>8} {'Max_RE':>8} {'Min_RE':>8} {'Δ vs scratch':>14}")
print("-"*95)

rows = [
    ("Target=DS1", "Cross-transfer DS-2→DS-1", cross_B,    scratch_D1),
    ("Target=DS1", "No-transfer DS-1 scratch",  scratch_D1, None),
    ("Target=DS2", "Cross-transfer DS-1→DS-2", cross_A,    scratch_D2),
    ("Target=DS2", "No-transfer DS-2 scratch",  scratch_D2, None),
]

for exp_name in EXPERIMENTS:
    print(f"\n{exp_name}")
    for target_lbl, dir_lbl, data_dict, baseline_dict in rows:
        st = data_dict[exp_name]
        mre = st["mean_re"].mean()
        mxr = st["max_re"].max()
        mnr = st["min_re"].min()
        if baseline_dict is not None:
            base_mre = baseline_dict[exp_name]["mean_re"].mean()
            delta = base_mre - mre
            delta_str = f"{delta:+.2f}%"
        else:
            delta_str = "  (baseline)"
        print(f"  [{target_lbl}] {dir_lbl:<38} {mre:>8.2f} {mxr:>8.2f} {mnr:>8.2f} {delta_str:>14}")

print("\n" + "="*95)
print("(+Δ = cross-transfer outperforms scratch by that margin)")


SUMMARY: Cross-transfer vs No-transfer Scratch  (full target dataset, SEED=42)
Experiment             Direction                                 Mean_RE   Max_RE   Min_RE   Δ vs scratch
-----------------------------------------------------------------------------------------------

Transmission Gain
  [Target=DS1] Cross-transfer DS-2→DS-1                   1.95    12.25     0.00         -0.15%
  [Target=DS1] No-transfer DS-1 scratch                   1.80    12.81     0.00     (baseline)
  [Target=DS2] Cross-transfer DS-1→DS-2                   2.47    21.31     0.00         -0.64%
  [Target=DS2] No-transfer DS-2 scratch                   1.82    16.86     0.00     (baseline)

MCS
  [Target=DS1] Cross-transfer DS-2→DS-1                   1.94    12.16     0.00         -0.14%
  [Target=DS1] No-transfer DS-1 scratch                   1.81    12.58     0.00     (baseline)
  [Target=DS2] Cross-transfer DS-1→DS-2                   2.43    19.55     0.00         -0.57%
  [Target=DS2] No-tran